In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

import tensorflow as tf
import keras_tuner as kt
import tensorboard
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout,BatchNormalization,Input,LSTM,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

I0000 00:00:1786508519.537518   29305 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786508519.550449   29305 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786508521.005890   29305 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786508523.786052   29305 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [3]:
df_train = pd.read_csv(r'../data/cleaned/train.csv')
df_test = pd.read_csv(r'../data/cleaned/test.csv')
df_val = pd.read_csv(r'../data/cleaned/val.csv')

In [4]:
with open('target_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [5]:
def create_sequences(df,seq_length, target_col='PJME_MW',horizon=1):
    """
    Converts a scaled DataFrame into 3D sequence arrays for Keras models.
    """
    # Separate feature columns from Datetime
    feature_cols = [col for col in df.columns if col != 'Datetime']
    target_idx = feature_cols.index(target_col)
    
    data = df[feature_cols].values
    X, y = [], []
    
    for i in range(len(data) - seq_length - horizon + 1):
        X.append(data[i : i + seq_length, :])
        if horizon == 1:
            y.append(data[i + seq_length, target_idx])
        else:
            y.append(data[i + seq_length : i + seq_length + horizon, target_idx])
            
    return np.array(X), np.array(y)

In [6]:
SEQ_LEN = 168
HORIZON = 24

X_train168, y_train168 = create_sequences(df_train, seq_length=SEQ_LEN, horizon=HORIZON)
X_val168, y_val168     = create_sequences(df_val, seq_length=SEQ_LEN, horizon=HORIZON)
X_test168, y_test168   = create_sequences(df_test, seq_length=SEQ_LEN, horizon=HORIZON)

print("X_train:", X_train168.shape)
print("y_train:", y_train168.shape)

print("X_test:", X_test168.shape)
print("y_test:", y_test168.shape)

X_train: (101449, 168, 13)
y_train: (101449, 24)
X_test: (21589, 168, 13)
y_test: (21589, 24)


In [7]:
y_test_mw = scaler.inverse_transform(y_test168.reshape(-1, 1))

In [8]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

In [9]:
early_stop = EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True,verbose=1)

In [ ]:
#drop out 0.2

In [10]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

E0000 00:00:1786447609.561290  741653 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 76ms/step - loss: 0.0057 - mae: 0.0526 - val_loss: 0.0020 - val_mae: 0.0335
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 130s 82ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0019 - val_mae: 0.0313
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 79ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0018 - val_mae: 0.0303
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 137s 86ms/step - loss: 0.0016 - mae: 0.0298 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 80ms/step - loss: 0.0016 - mae: 0.0291 - val_loss: 0.0016 - val_mae: 0.0282
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 127s 80ms/step - loss: 0.0015 - mae: 0.0285 - val_loss: 0.0016 - val_mae: 0.0286
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 124s 78ms/step - loss: 0.0015 - mae: 0.0283 - val_loss: 0.0016 - val_mae: 0.0287
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 119s 75ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0016 - val_mae: 0.0287
Epoch 9/10
1586/

In [11]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1483.72 MW
RMSE: 2104.47 MW
MAPE: 4.57%
R2:   0.8932


In [ ]:
#0.3

In [12]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 126s 78ms/step - loss: 0.0064 - mae: 0.0565 - val_loss: 0.0020 - val_mae: 0.0330
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0021 - mae: 0.0346 - val_loss: 0.0018 - val_mae: 0.0317
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 117s 74ms/step - loss: 0.0019 - mae: 0.0325 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 119s 75ms/step - loss: 0.0018 - mae: 0.0316 - val_loss: 0.0016 - val_mae: 0.0293
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0017 - mae: 0.0308 - val_loss: 0.0019 - val_mae: 0.0309
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 118s 74ms/step - loss: 0.0017 - mae: 0.0304 - val_loss: 0.0020 - val_mae: 0.0330
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 131s 83ms/step - loss: 0.0016 - mae: 0.0299 - val_loss: 0.0015 - val_mae: 0.0281
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 133s 84ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0015 - val_mae: 0.0286
Epoch 9/10
1586/

In [13]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1469.83 MW
RMSE: 2007.76 MW
MAPE: 4.66%
R2:   0.9028


In [14]:
model_bilstm.save(r'../models/drop_bilstm.keras')
history_df =pd.DataFrame(history_bilstm.history)

history_df.to_csv(r'../log/drop_bilstm.csv',index=False)

In [ ]:
#0.5

In [15]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    verbose=1
)

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 123s 76ms/step - loss: 0.0079 - mae: 0.0622 - val_loss: 0.0025 - val_mae: 0.0380
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0027 - mae: 0.0394 - val_loss: 0.0023 - val_mae: 0.0373
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 123s 77ms/step - loss: 0.0023 - mae: 0.0362 - val_loss: 0.0018 - val_mae: 0.0316
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0022 - mae: 0.0350 - val_loss: 0.0018 - val_mae: 0.0314
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 122s 77ms/step - loss: 0.0021 - mae: 0.0343 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 120s 76ms/step - loss: 0.0020 - mae: 0.0337 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 128s 81ms/step - loss: 0.0020 - mae: 0.0331 - val_loss: 0.0016 - val_mae: 0.0295
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 135s 85ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0016 - val_mae: 0.0291
Epoch 9/10
1586/

In [16]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1500.72 MW
RMSE: 2065.52 MW
MAPE: 4.72%
R2:   0.8971


In [ ]:
#early stop

In [10]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    callbacks=[early_stop]
)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 422s 259ms/step - loss: 0.0071 - mae: 0.0604 - val_loss: 0.0024 - val_mae: 0.0373
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 405s 255ms/step - loss: 0.0026 - mae: 0.0387 - val_loss: 0.0019 - val_mae: 0.0321
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 1159s 731ms/step - loss: 0.0022 - mae: 0.0355 - val_loss: 0.0017 - val_mae: 0.0305
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 179s 113ms/step - loss: 0.0021 - mae: 0.0344 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 121s 76ms/step - loss: 0.0020 - mae: 0.0337 - val_loss: 0.0017 - val_mae: 0.0300
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 111s 70ms/step - loss: 0.0019 - mae: 0.0330 - val_loss: 0.0016 - val_mae: 0.0294
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 108s 68ms/step - loss: 0.0019 - mae: 0.0326 - val_loss: 0.0016 - val_mae: 0.0292
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 96s 61ms/step - loss: 0.0019 - mae: 0.0321 - val_loss: 0.0016 - val_mae: 0.0294
Epoch 9/10
1

In [11]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step
MAE:  1488.85 MW
RMSE: 2028.45 MW
MAPE: 4.75%
R2:   0.9008


In [ ]:
#early stopping

In [12]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
    callbacks=[lr_scheduler]
)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 100s 61ms/step - loss: 0.0081 - mae: 0.0636 - val_loss: 0.0026 - val_mae: 0.0390 - learning_rate: 0.0010
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 98s 62ms/step - loss: 0.0026 - mae: 0.0389 - val_loss: 0.0019 - val_mae: 0.0322 - learning_rate: 0.0010
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 111s 70ms/step - loss: 0.0023 - mae: 0.0359 - val_loss: 0.0018 - val_mae: 0.0311 - learning_rate: 0.0010
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 118s 75ms/step - loss: 0.0022 - mae: 0.0350 - val_loss: 0.0019 - val_mae: 0.0310 - learning_rate: 0.0010
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 130s 82ms/step - loss: 0.0021 - mae: 0.0345 - val_loss: 0.0016 - val_mae: 0.0296 - learning_rate: 0.0010
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 144s 91ms/step - loss: 0.0021 - mae: 0.0341 - val_loss: 0.0016 - val_mae: 0.0296 - learning_rate: 0.0010
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - loss: 0.0020 - mae: 0.0337
Epoch 7: ReduceLROnPlateau reducing

In [13]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step
MAE:  1457.88 MW
RMSE: 2001.02 MW
MAPE: 4.64%
R2:   0.9035


In [14]:
model_bilstm.save(r'../models/bilstm_lr.keras')
history_df =pd.DataFrame(history_bilstm.history)

history_df.to_csv(r'../log/lr_bilstm.csv',index=False)

In [ ]:
#batch normalization

In [15]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    BatchNormalization(),
    Dropout(0.5),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 105s 63ms/step - loss: 0.0462 - mae: 0.1275 - val_loss: 0.0043 - val_mae: 0.0516
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 101s 64ms/step - loss: 0.0052 - mae: 0.0565 - val_loss: 0.0033 - val_mae: 0.0446
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 112s 70ms/step - loss: 0.0038 - mae: 0.0476 - val_loss: 0.0024 - val_mae: 0.0372
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 110s 69ms/step - loss: 0.0032 - mae: 0.0435 - val_loss: 0.0022 - val_mae: 0.0354
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 108s 68ms/step - loss: 0.0029 - mae: 0.0412 - val_loss: 0.0021 - val_mae: 0.0341
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 113s 71ms/step - loss: 0.0027 - mae: 0.0395 - val_loss: 0.0020 - val_mae: 0.0337
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 106s 67ms/step - loss: 0.0026 - mae: 0.0385 - val_loss: 0.0019 - val_mae: 0.0333
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 107s 67ms/step - loss: 0.0025 - mae: 0.0379 - val_loss: 0.0018 - val_mae: 0.0313
Epoch 9/10
1586/

In [ ]:
#rmsprop

In [18]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='RMSprop', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 102s 63ms/step - loss: 0.0053 - mae: 0.0522 - val_loss: 0.0038 - val_mae: 0.0469
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 98s 62ms/step - loss: 0.0022 - mae: 0.0357 - val_loss: 0.0029 - val_mae: 0.0426
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 99s 62ms/step - loss: 0.0019 - mae: 0.0322 - val_loss: 0.0024 - val_mae: 0.0379
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 99s 62ms/step - loss: 0.0017 - mae: 0.0307 - val_loss: 0.0019 - val_mae: 0.0321
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 98s 62ms/step - loss: 0.0016 - mae: 0.0298 - val_loss: 0.0022 - val_mae: 0.0346
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 99s 62ms/step - loss: 0.0016 - mae: 0.0292 - val_loss: 0.0020 - val_mae: 0.0327
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 99s 63ms/step - loss: 0.0015 - mae: 0.0287 - val_loss: 0.0018 - val_mae: 0.0311
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 98s 62ms/step - loss: 0.0015 - mae: 0.0284 - val_loss: 0.0019 - val_mae: 0.0319
Epoch 9/10
1586/1586 ━━

In [19]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step
MAE:  1849.57 MW
RMSE: 2480.93 MW
MAPE: 6.05%
R2:   0.8516


In [ ]:
#batch size

In [21]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=32,
)

Epoch 1/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 129s 39ms/step - loss: 0.0033 - mae: 0.0384 - val_loss: 0.0020 - val_mae: 0.0326
Epoch 2/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 126s 40ms/step - loss: 0.0015 - mae: 0.0286 - val_loss: 0.0017 - val_mae: 0.0301
Epoch 3/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 127s 40ms/step - loss: 0.0014 - mae: 0.0272 - val_loss: 0.0017 - val_mae: 0.0289
Epoch 4/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 126s 40ms/step - loss: 0.0013 - mae: 0.0264 - val_loss: 0.0015 - val_mae: 0.0276
Epoch 5/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 126s 40ms/step - loss: 0.0013 - mae: 0.0255 - val_loss: 0.0015 - val_mae: 0.0274
Epoch 6/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 126s 40ms/step - loss: 0.0012 - mae: 0.0248 - val_loss: 0.0017 - val_mae: 0.0299
Epoch 7/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 127s 40ms/step - loss: 0.0012 - mae: 0.0242 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 8/10
3171/3171 ━━━━━━━━━━━━━━━━━━━━ 126s 40ms/step - loss: 0.0011 - mae: 0.0236 - val_loss: 0.0017 - val_mae: 0.0288
Epoch 9/10
3171/

In [22]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step
MAE:  1448.90 MW
RMSE: 2061.29 MW
MAPE: 4.63%
R2:   0.8976


In [ ]:
#16

In [11]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=False), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=16,
)

c:\Users\rizwa\.virtualenvs\week_8-5qCf5Pni\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 194s 30ms/step - loss: 0.0029 - mae: 0.0373 - val_loss: 0.0017 - val_mae: 0.0296
Epoch 2/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 189s 30ms/step - loss: 0.0015 - mae: 0.0279 - val_loss: 0.0019 - val_mae: 0.0320
Epoch 3/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 200s 32ms/step - loss: 0.0014 - mae: 0.0265 - val_loss: 0.0015 - val_mae: 0.0277
Epoch 4/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 201s 32ms/step - loss: 0.0013 - mae: 0.0254 - val_loss: 0.0017 - val_mae: 0.0294
Epoch 5/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 205s 32ms/step - loss: 0.0012 - mae: 0.0246 - val_loss: 0.0014 - val_mae: 0.0270
Epoch 6/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 250s 30ms/step - loss: 0.0011 - mae: 0.0239 - val_loss: 0.0015 - val_mae: 0.0273
Epoch 7/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 202s 32ms/step - loss: 0.0011 - mae: 0.0235 - val_loss: 0.0015 - val_mae: 0.0268
Epoch 8/10
6341/6341 ━━━━━━━━━━━━━━━━━━━━ 204s 32ms/step - loss: 0.0010 - mae: 0.0229 - val_loss: 0.0016 - val_mae: 0.0273
Epoch 9/10
6341/

In [13]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step
MAE:  1507.54 MW
RMSE: 2127.13 MW
MAPE: 4.76%
R2:   0.8909


In [ ]:
#multiple layers

In [15]:
model_bilstm = Sequential([
    Bidirectional(LSTM(64, return_sequences=True), input_shape=(X_train168.shape[1], X_train168.shape[2])),
    Dropout(0.2),
    
    Bidirectional(LSTM(32,return_sequences=False)),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dense(HORIZON)
])

model_bilstm.compile(optimizer='adam', loss='mse', metrics=['mae'])


history_bilstm = model_bilstm.fit(
    X_train168, y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 194s 119ms/step - loss: 0.0065 - mae: 0.0546 - val_loss: 0.0022 - val_mae: 0.0359
Epoch 2/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 191s 121ms/step - loss: 0.0021 - mae: 0.0348 - val_loss: 0.0018 - val_mae: 0.0319
Epoch 3/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 191s 120ms/step - loss: 0.0019 - mae: 0.0324 - val_loss: 0.0017 - val_mae: 0.0301
Epoch 4/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 193s 121ms/step - loss: 0.0018 - mae: 0.0313 - val_loss: 0.0018 - val_mae: 0.0311
Epoch 5/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 192s 121ms/step - loss: 0.0017 - mae: 0.0306 - val_loss: 0.0016 - val_mae: 0.0293
Epoch 6/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 188s 119ms/step - loss: 0.0016 - mae: 0.0300 - val_loss: 0.0016 - val_mae: 0.0291
Epoch 7/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 179s 113ms/step - loss: 0.0016 - mae: 0.0295 - val_loss: 0.0016 - val_mae: 0.0290
Epoch 8/10
1586/1586 ━━━━━━━━━━━━━━━━━━━━ 178s 112ms/step - loss: 0.0015 - mae: 0.0290 - val_loss: 0.0016 - val_mae: 0.0283
Epoch 9/

In [16]:
y_pred_bilstm_scaled = model_bilstm.predict(X_test168)

y_pred_bilstm_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_bilstm  = mean_absolute_error(y_test_mw, y_pred_bilstm_mw)
rmse_bilstm = np.sqrt(mean_squared_error(y_test_mw, y_pred_bilstm_mw))
mape_bilstm = np.mean(np.abs((y_test_mw - y_pred_bilstm_mw) / y_test_mw)) * 100
r2_bilstm  = r2_score(y_test_mw, y_pred_bilstm_mw)

print(f"MAE:  {mae_bilstm:.2f} MW")
print(f"RMSE: {rmse_bilstm:.2f} MW")
print(f"MAPE: {mape_bilstm:.2f}%")
print(f"R2:   {r2_bilstm:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step
MAE:  1646.47 MW
RMSE: 2257.01 MW
MAPE: 5.12%
R2:   0.8772


In [ ]:
#hyper parameter

In [10]:
input_shape = (X_train168.shape[1], X_train168.shape[2])

def build_model(hp):

    model = Sequential([
        Input(shape=input_shape),

        Bidirectional(LSTM(
            units=hp.Choice(
                "bilstm_units",
                [32, 64, 128]
            )
        )),

        BatchNormalization(),

        Dense(
            units=hp.Choice(
                "dense_units",
                [16, 32, 64]
            ),
            activation="relu"
        ),

        Dropout(
            hp.Choice(
                "dropout",
                [0.2, 0.3, 0.5]
            )
        ),

        Dense(HORIZON)
    ])

    model.compile(
         optimizer=hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd"]),
        loss="mse",
        metrics=["mae"]
    )

    return model

In [11]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_loss",
    max_trials=5,
    directory="enn_tuning",
    project_name="bilstm_forecasting"
)

E0000 00:00:1786508566.755134   29305 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [12]:
tuner.search(
    X_train168,
    y_train168,
    validation_data=(X_val168, y_val168),
    epochs=10,
    batch_size=64
)

Trial 5 Complete [00h 48m 49s]
val_loss: 0.003665370400995016

Best val_loss So Far: 0.0020759489852935076
Total elapsed time: 02h 47m 08s


In [13]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("bilstm units:", best_hp.get("bilstm_units"))
print("Dense units:", best_hp.get("dense_units"))
print("Dropout:", best_hp.get("dropout"))

bilstm units: 128
Dense units: 64
Dropout: 0.3


In [14]:
best_model = tuner.get_best_models(num_models=1)[0]

/home/aximsoft/snap/code/255/.local/share/virtualenvs/week_8-_hg6qRLY/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


In [15]:
y_pred_bilstm_scaled = best_model.predict(X_test168)

y_pred_rnn_mw = scaler.inverse_transform(y_pred_bilstm_scaled.reshape(-1, 1))

mae_rnn  = mean_absolute_error(y_test_mw, y_pred_rnn_mw)
rmse_rnn = np.sqrt(mean_squared_error(y_test_mw, y_pred_rnn_mw))
mape_rnn = np.mean(np.abs((y_test_mw - y_pred_rnn_mw) / y_test_mw)) * 100
r2_rnn  = r2_score(y_test_mw, y_pred_rnn_mw)

print(f"MAE:  {mae_rnn:.2f} MW")
print(f"RMSE: {rmse_rnn:.2f} MW")
print(f"MAPE: {mape_rnn:.2f}%")
print(f"R2:   {r2_rnn:.4f}")

675/675 ━━━━━━━━━━━━━━━━━━━━ 19s 28ms/step
MAE:  1671.14 MW
RMSE: 2231.40 MW
MAPE: 5.39%
R2:   0.8800
